# Simple linear baselines

In [1]:
import re
import os
import random
import pandas as pd
from statsmodels.formula.api import ols
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [2]:
def split_mut_sit(mutation):
    parts = re.match(r'([A-Za-z])(\d+)(\S)', mutation) # \S for any non- white space
    if parts:
        return list(parts.groups())
    else:
        return [None, None, None]

In [55]:
#data = pd.read_csv('../data/viral/metadata/SARS2_DELTA_SPIKE_Dadonaite.csv')
data = pd.read_csv('../data/viral/metadata/IAV_H1_HA_Wu.csv')
#data = pd.read_csv('../data/viral/metadata/SARS2_RBD_binding_Starr.csv')
data[['wt', 'site', 'mut']] = data['mutant'].apply(lambda x: pd.Series(split_mut_sit(x)))
data['target'] = scaler.fit_transform(data['target'].to_frame()).squeeze()
data['site'] = data['site'].astype(str)

common_sites = data['site'].value_counts()[data['site'].value_counts() > 4].index
data = data[data['site'].isin(common_sites)]

## Random split

In [ ]:
train_data, test_data = train_test_split(data, test_size=0.15, random_state=4)

# Fit the OLS model using formula interface
model = ols('target ~ site', data=train_data).fit()

# Make predictions on test set
y = test_data.copy()
y['target'] = 0
test_predictions = model.predict(y)

# Calculate performance metrics
test_r2 = metrics.r2_score(test_data['target'], test_predictions)

print(f"Test Set Performance:")
print(f"R² Score: {test_r2:.3f}")

In [57]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=4, stratify=data['site'])

# Fit the OLS model using formula interface
model = ols('target ~ site', data=train_data).fit()

# Make predictions on test set
y = test_data.copy()
y['target'] = 0
test_predictions = model.predict(y)

# Calculate performance metrics
test_r2 = metrics.r2_score(test_data['target'], test_predictions)

print(f"Test Set Performance:")
print(f"R² Score: {test_r2:.3f}")

Test Set Performance:
R² Score: -0.111


## Visualize results

In [62]:
import os

full_res = pd.DataFrame()
for source in ['viral', 'nonviral']:
    res_dir = f'../experiments/OLS/{source}/pool_split/'
    for file in os.listdir(res_dir):
        dataset = file.split('.csv')[0]
        df = pd.read_csv(os.path.join(res_dir, file), index_col=0)
        df['Dataset'] = dataset
        df['Source'] = source
        df['Model'] = 'OLS'
        df['Split'] = 'pooled'
        full_res = pd.concat([full_res, df])
        
full_res

,Model,Fold,R2_score_train,MAE_score_train,RMSE_score_train,R2_score_test,MAE_score_test,RMSE_score_test,rho_score_train,rho_score_test,Dataset,Source,Split
0,OLS,1,0.376406,0.628156,0.792004,0.304368,0.658004,0.824147,0.598880,0.540495,IAV_H1_NP_Doud,viral,pooled
1,OLS,2,0.379339,0.625441,0.787427,0.290447,0.672475,0.844004,0.601615,0.526581,IAV_H1_NP_Doud,viral,pooled
2,OLS,3,0.378776,0.625165,0.787603,0.300951,0.671381,0.838504,0.600995,0.533578,IAV_H1_NP_Doud,viral,pooled
0,OLS,1,0.348981,0.343840,0.762675,0.203165,0.379728,0.641082,0.639177,0.371428,IAV_H1_HA_Wu,viral,pooled
1,OLS,2,0.389340,0.328689,0.711756,-0.026567,0.398681,0.890459,0.630545,0.419289,IAV_H1_HA_Wu,viral,pooled
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,OLS,2,0.137988,0.731342,0.926392,-0.026640,0.798818,1.022744,0.368474,0.140475,CALM1_HUMAN_Roth2017,nonviral,pooled
2,OLS,3,0.141814,0.734375,0.932268,-0.024135,0.785742,0.986381,0.369130,0.177240,CALM1_HUMAN_Roth2017,nonviral,pooled
0,OLS,1,0.509198,0.412195,0.684900,0.448029,0.479561,0.804435,0.718037,0.682826,IF1_ECOLI,nonviral,pooled
1,OLS,2,0.490962,0.425552,0.704523,0.502878,0.454994,0.738491,0.720461,0.700028,IF1_ECOLI,nonviral,pooled


In [63]:
full_res.to_csv('../experiments_v01/lassoCV/results_OLS_3folds_v_nv_Pooled.csv')

## Site split

In [39]:
def split_data(meta_data, seed, train_pct=0.8, test_pct=0.2):
    # find sites of mutation and order randomly
    meta_data["site"] = [int(s[1:-1]) for s in meta_data["mutant"]]
    sites = meta_data["site"].unique()
    random.seed(seed)
    random.shuffle(sites)

    if train_pct + test_pct != 1:
        print("Split percentages must sum to 1")
        return

    df_size = meta_data.shape[0]
    df_test_size = df_size*test_pct
    test_sites, train_sites = [], []

    # determine sites for test, then train
    for site in sites:
        if len(test_sites) <= df_test_size:
            test_sites.extend([mut_site for mut_site in meta_data["site"] if mut_site == site])
        else:
            train_sites.extend([mut_site for mut_site in meta_data["site"] if mut_site == site])

    # subset df for train, test data
    train_df = meta_data[meta_data["site"].isin(set(train_sites))]
    test_df = meta_data[meta_data["site"].isin(set(test_sites))]

    return train_df, test_df

In [ ]:
train_data, test_data = split_data(data, 4)
train_data['site'] = train_data['site'].astype(str)
test_data['site'] = test_data['site'].astype(str)

# Fit the OLS model using formula interface
model = ols('target ~ site', data=train_data).fit()

# Make predictions on test set
y = test_data.copy()
y['target'] = 0
test_predictions = model.predict(y)

# Calculate performance metrics
test_r2 = metrics.r2_score(test_data['target'], test_predictions)

print(f"Test Set Performance:")
print(f"R² Score: {test_r2:.3f}")